In [1]:
year = 1993
month = 2

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap

### URLs

In [3]:
SSHfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-grid2D"
#mesh url
Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [4]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [5]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [6]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [7]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [8]:
# ds = xr.open_dataset(SSHfiles, engine="pydap", mask_and_scale=False, decode_cf=True).sossheig.sortby("time_counter").isel(time_counter=0)
# # ds = ds.where(ds!= 9.96921e+36,np.nan)
# ds

<xarray.DataArray 'sossheig' (y: 3059, x: 4322)> Size: 53MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(3059, 4322), dtype=float32)
Coordinates:
  * x             (x) int32 17kB 1 2 3 4 5 6 7 ... 4317 4318 4319 4320 4321 4322
  * y             (y) int32 12kB 1 2 3 4 5 6 7 ... 3054 3055 3056 3057 3058 3059
    time_counter  datetime64[ns] 8B 1992-12-30T12:00:00
    nav_lon       (y, x) float32 53MB 72.92 73.0 73.08 73.17 ... 73.0 73.0 73.0
    nav_lat       (y, x) float32 53MB -77.01 -77.01 -77.01 ... 50.0 50.0 50.0
Attributes: (12/15)
    units:               m
    missing_value:       9.96921e+36
    _FillValue:          9.96921e+36
    valid_min:           -10.0
    valid_max:           10.0
    add_offset:          0.0
    ...                  ...
    short_name:          sossheig
    online_operation:    N/A
    interval_operation:  86400
    interval_write:      86400
    associate:           time_counter nav_lat nav_lon
    _ChunkSizes:         [1, 765, 1081]

In [9]:
import calendar
import datetime
from datetime import date

In [10]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1993-02-28


In [11]:
import pandas as pd

def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

# Example
days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [12]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        # da = da.where(ds!= 9.96921e+36,np.nan)
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all = da_all.where(da_all!= 9.96921e+36,np.nan)
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [13]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1993-02-01 12:00:00
end_date 1993-02-02 12:00:00
start_date 1993-02-03 12:00:00
end_date 1993-02-04 12:00:00
start_date 1993-02-05 12:00:00
end_date 1993-02-06 12:00:00
start_date 1993-02-07 12:00:00
end_date 1993-02-08 12:00:00
start_date 1993-02-09 12:00:00
end_date 1993-02-10 12:00:00
start_date 1993-02-11 12:00:00
end_date 1993-02-12 12:00:00
start_date 1993-02-13 12:00:00
end_date 1993-02-14 12:00:00
start_date 1993-02-15 12:00:00
end_date 1993-02-16 12:00:00
start_date 1993-02-17 12:00:00
end_date 1993-02-18 12:00:00
start_date 1993-02-19 12:00:00
end_date 1993-02-20 12:00:00
start_date 1993-02-21 12:00:00
end_date 1993-02-22 12:00:00
start_date 1993-02-23 12:00:00
end_date 1993-02-24 12:00:00
start_date 1993-02-25 12:00:00
end_date 1993-02-26 12:00:00
start_date 1993-02-27 12:00:00
end_date 1993-02-28 12:00:00


### Data download

In [14]:
SSH_out = f'SSH_{start_date.strftime("%Y-%m-%d")[:7]}c.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables_c/'

In [15]:
download_MERCATOR(
    SSHfiles, "sossheig", starts, ends, x0, x1, y0, y1,outpath+SSH_out 
)

100%|██████████| 14/14 [01:05<00:00,  4.64s/it]


Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_1993-02.nc
